In [23]:
import os
import re

import pandas as pd

raw_dir = "../data/raw/"
patch_dir = "../data/patches/"
os.makedirs(patch_dir, exist_ok=True)

targets = {
    "LEHMQ": {
        "file": "LEHMQ_daily.html",
        "Name": "Lehman Brothers",
        "Category": "Distressed_2008",
        "Sector": "Financials",
        "max_date": "2008-09-15",
    },
    "BSC": {
        "file": "BSC_daily.html",
        "Name": "Bear Stearns",
        "Category": "Distressed_2008",
        "Sector": "Financials",
        "max_date": "2008-03-16",
    },
    "SIVB": {
        "file": "SVB_daily.html",
        "Name": "Silicon Valley Bank",
        "Category": "Distressed_2023",
        "Sector": "Financials",
        "max_date": "2023-03-10",
    },
    "SBNY": {
        "file": "SBNY_daily.html",
        "Name": "Signature Bank",
        "Category": "Distressed_2023",
        "Sector": "Financials",
        "max_date": "2023-03-10",
    },
    "PACW": {
        "file": "PACW_daily.html",
        "Name": "PacWest Bancorp",
        "Category": "Distressed_2023",
        "Sector": "Financials",
        "max_date": "2023-05-10",
    },
    "CHK": {
        "file": "CHK_daily.html",
        "Name": "Chesapeake Energy",
        "Category": "Distressed_2020",
        "Sector": "Energy",
        "max_date": "2020-06-30",
    },
}

print("--- PARSING & TRUNCATING HISTORICAL CRISIS WINDOWS ---")

for ticker, meta in targets.items():
    html_path = os.path.join(raw_dir, meta["file"])

    if os.path.exists(html_path):
        with open(html_path, "r", encoding="utf-8") as f:
            content = f.read()

        records = []
        # Flexible regex to grab Unix timestamps ('d') and values ('v')
        pattern = r'"d"\s*:\s*(\d+)\s*,\s*"v"\s*:\s*"?([0-9\.]+)"?'
        matches = re.findall(pattern, content)

        if matches:
            for d_val, v_val in matches:
                dt = pd.to_datetime(int(d_val), unit="s")
                val = float(v_val)
                records.append({"Date": dt, "Close": val, "Adj_Close": val})

            df_entity = pd.DataFrame(records)
            df_entity = df_entity.sort_values("Date").drop_duplicates(subset=["Date"])

            # Truncate data strictly to active operational life (dropping post-bankruptcy noise)
            if "max_date" in meta:
                df_entity = df_entity[
                    df_entity["Date"] <= pd.to_datetime(meta["max_date"])
                ]

            df_entity["Ticker"] = ticker
            df_entity["Name"] = meta["Name"]
            df_entity["Category"] = meta["Category"]
            df_entity["Sector"] = meta["Sector"]

            df_entity["Open"] = df_entity["Close"]
            df_entity["High"] = df_entity["Close"]
            df_entity["Low"] = df_entity["Close"]
            df_entity["Volume"] = 0

            df_entity = df_entity[
                [
                    "Date",
                    "Ticker",
                    "Name",
                    "Category",
                    "Sector",
                    "Open",
                    "High",
                    "Low",
                    "Close",
                    "Adj_Close",
                    "Volume",
                ]
            ]

            output_csv = os.path.join(patch_dir, f"{ticker}.csv")
            df_entity.to_csv(output_csv, index=False)
            print(
                f" ✅ [Success] {meta['Name']} ({ticker}): Parsed {len(df_entity):,} pre-collapse"
                f" rows up to {meta['max_date']}"
            )
        else:
            print(f" ❌ [Warning] Could not match chart pattern in {meta['file']}")
    else:
        print(f" ❌ [Notice] File not found: {html_path}")

print(
    "\nPatch truncation complete! Run your 01_data_ingestion.ipynb pipeline to"
    " build the master dataset."
)

--- PARSING & TRUNCATING HISTORICAL CRISIS WINDOWS ---
 ✅ [Success] Lehman Brothers (LEHMQ): Parsed 172 pre-collapse rows up to 2008-09-15
 ✅ [Success] Bear Stearns (BSC): Parsed 218 pre-collapse rows up to 2008-03-16
 ✅ [Success] Silicon Valley Bank (SIVB): Parsed 425 pre-collapse rows up to 2023-03-10
 ✅ [Success] Signature Bank (SBNY): Parsed 0 pre-collapse rows up to 2023-03-10
 ✅ [Success] PacWest Bancorp (PACW): Parsed 275 pre-collapse rows up to 2023-05-10
 ✅ [Success] Chesapeake Energy (CHK): Parsed 0 pre-collapse rows up to 2020-06-30

Patch truncation complete! Run your 01_data_ingestion.ipynb pipeline to build the master dataset.
